# Day 4: Streaming, data quality and governance

**Modern Data Engineering for AI Systems · SDA-DSC-214**

Receive real Kafka events into Delta, restart a query with the same checkpoint, and distinguish delivery records from business events. Then reject, quarantine and revalidate an intentionally flawed candidate.

**Your lab:** 45 minutes to run and check, 5 minutes to answer, 5 minutes to discuss, and 5 minutes to save.

[Today's guide](https://github.com/almiyead-rgb/masar-ai-data-engineering-ksu/blob/main/day04/README.md) · [Expected results](https://github.com/almiyead-rgb/masar-ai-data-engineering-ksu/tree/main/expected) · [Submission guide](https://github.com/almiyead-rgb/masar-ai-data-engineering-ksu/blob/main/submission/README.md)

## Before you run

Use a CPU runtime. Select the documented Python 3.12 Colab runtime before running the first cell. The setup checks the versions and prepares Java 17. If Colab asks to restart after package installation, reconnect and run the setup again. Keep one GitHub repository for all five days.

Run cells in order. A dependency check is not a completed lab. Compare the outputs produced below with the reference results after the real engines run.

In [ ]:
from pathlib import Path
import importlib.util, json, os, runpy, subprocess, sys
DAY = 4
# Keep your own fork's owner/name here if you want to load committed edits.
PROJECT_REPOSITORY = "almiyead-rgb/masar-ai-data-engineering-ksu"
NONINTERACTIVE = os.environ.get("MASAR_NONINTERACTIVE") == "1"
try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False
if IS_COLAB and not NONINTERACTIVE:
    ROOT = Path("/content/masar-project")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/" + PROJECT_REPOSITORY + ".git", str(ROOT)], check=True)
    if not (ROOT / "course.json").is_file():
        raise RuntimeError("The project folder is incomplete. Preserve it and inspect the setup guide.")
    os.chdir(ROOT)
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "course.json").is_file() and (p / "src/masar").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Open this notebook from the complete course repository.")
sys.path.insert(0, str(ROOT / "src"))
bootstrap = runpy.run_path(str(ROOT / "scripts/bootstrap_colab.py"))
environment = bootstrap["prepare"](install=IS_COLAB)
os.environ["JAVA_HOME"] = environment["java_home"]
print(json.dumps(environment, indent=2))
from masar.runtime import start_spark
from masar.workspace import require_fixed_dataset, completed_bronze_workspace, new_workspace, write_json
from masar.native_contracts import validate_stage_result
from masar.colab import restore_handoff, record_reflection, record_metrics, save_day, initialize_student_notes
SOURCE = ROOT / "data/masar-small-v1"
require_fixed_dataset(SOURCE)
print("Course source:", ROOT)
print("Dependency checks are complete. The following cells execute the real engines.")
print("Interactive Drive/download UI enabled:", IS_COLAB and not NONINTERACTIVE)


## Your saved project

Allow Colab to mount your own Drive. Your daily continuation archives are saved in `MyDrive/Masar_Project`. A new session restores the preceding day only when the local project has no saved workspace. Existing work is not overwritten.

If you uploaded a handoff ZIP manually, call `restore_handoff` with its path and the preceding day number. Keep earlier archives while you learn.

After restoration, this cell creates your lab-note prompts under `student/dayNN/` and the shared project notes under `student/`. Open the printed paths in Colab’s Files panel and add your own evidence. Existing files are preserved. Your daily question answer belongs in `student/dayNN/REFLECTION.md`; the final save cell includes all these files in both packages.

In [ ]:
DRIVE_FOLDER = None
if IS_COLAB and not NONINTERACTIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_FOLDER = Path("/content/drive/MyDrive/Masar_Project")
    DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)
if DAY > 1:
    pointer = ROOT / "outputs/day01_bronze_success.json"
    if not pointer.exists():
        backup = DRIVE_FOLDER / f"day{DAY-1:02d}_handoff.zip" if DRIVE_FOLDER else None
        if backup is None or not backup.is_file():
            raise FileNotFoundError(
                f"Restore your day{DAY-1:02d}_handoff.zip before continuing. "
                "Use restore_handoff(ROOT, Path('/path/to/your/archive.zip'), expected_day=DAY-1).")
        print(json.dumps(restore_handoff(ROOT, backup, expected_day=DAY-1), indent=2))
    WORK = completed_bronze_workspace(ROOT)
    print("Continue your project:", WORK.relative_to(ROOT))
else:
    pointer = ROOT / "outputs/day01_bronze_success.json"
    if pointer.exists():
        existing = completed_bronze_workspace(ROOT)
        if (existing / "reports/day02_silver.json").exists():
            raise RuntimeError("Your project has advanced beyond Day 1. Open the next day's notebook. Use a separate clean project for a complete replay.")
        WORK = existing
        REUSE_BRONZE = True
        print("Reusing your completed Bronze workspace without re-ingesting the same batches.")
    else:
        WORK = new_workspace(ROOT, "day01_bronze")
        REUSE_BRONZE = False
        print("New project:", WORK.relative_to(ROOT))

# Create prompts only after restoring your existing work. Never overwrite a note.
notes = initialize_student_notes(ROOT, DAY)
print(json.dumps(notes, indent=2))


## Five-minute activity

Two Kafka messages have different offsets but the same event_id and identical payloads. How many transport receipts and distinct business events are present?

<details><summary>Discuss the answer after trying</summary>

Two transport receipts and one distinct business event. Offset identity is different from business-event identity.

</details>

## Prepare the local Kafka broker

Start the official Kafka distribution as a local Java process. The broker listens only inside this runtime. Its API health check must pass before the stream exercise.

If a VM was deleted, the previous Kafka broker no longer exists. Use the saved Day 3 state and run this exercise again with its new topic. Do not reconnect an old checkpoint to a newly created topic.

In [ ]:
from masar.kafka_local import start_kafka, stop_kafka
broker = start_kafka(ROOT)
print(json.dumps(broker, indent=2))

## Lab 05: messages, events and checkpoints

The exercise uses a new topic for this attempt and the same checkpoint across its four phases. Expected transport counts: **216, 216, 218, 219**. Expected unique event counts: **216, 216, 216, 217**.

In [ ]:
from masar.streaming import run_stream_lab
spark = start_spark(WORK, kafka=True)
try:
    result = run_stream_lab(spark, SOURCE, WORK)
    validate_stage_result('lab05_streaming', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('Transport rows:', [phase['transport_rows'] for phase in result['phases']])
    print('Unique event IDs:', [phase['unique_event_ids'] for phase in result['phases']])
    spark.read.format('delta').load(str(WORK/result['event_table'])).select('event_id','trip_id','event_ts').orderBy('event_id').show(5, truncate=False)
    metrics = {"transport_receipts": result["phases"][-1]["transport_rows"],
               "unique_events": result["phases"][-1]["unique_event_ids"]}
    # Your turn: choose the event identity, then compare receipts with distinct events.
    event_key = "event_id"
    actual_events = spark.read.format("delta").load(str(WORK / result["event_table"]))
    print("Distinct business-event keys:", actual_events.select(event_key).distinct().count())
finally:
    spark.stop()

## Lab 06: reject, quarantine and revalidate

The clean batch should pass. The 82-row mixed candidate should fail. Quarantine the seven invalid rows with reasons, then validate the 75-row candidate before approving it. Inspect both the GX results and the actual Delta tables.

In [ ]:
from masar.quality_gate import run_quality_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_quality_lab(spark, SOURCE, WORK)
    validate_stage_result('lab06_quality', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('Quarantined records:')
    spark.read.format('delta').load(str(WORK/result['quarantine_table'])).show(7, truncate=False)
    print('Approved rows:', spark.read.format('delta').load(str(WORK/result['approved_table'])).count())
    from pyspark.sql import functions as F
    approved = spark.read.format("delta").load(str(WORK / result["approved_table"]))
    metrics.update(candidate_rows=spark.read.format("delta").load(str(WORK / result["candidate_table"])).count(),
        quarantined_rows=spark.read.format("delta").load(str(WORK / result["quarantine_table"])).count(),
        approved_rows=approved.count(), total_fare_sar=format(approved.agg(F.sum("fare_sar")).first()[0], ".2f"))
finally:
    spark.stop()

record_metrics(ROOT, DAY, metrics)
print(json.dumps(stop_kafka(ROOT), indent=2))

## End-of-lab question

A candidate contains 82 rows and fails validation because seven rows are invalid. What must happen before the remaining 75 rows can be approved?

Write your answer and one observed value before opening the model answer. Your reflection is part of the project, worth one point when it explains your own run.

In [ ]:
MY_ANSWER = ""  # Write your own explanation before viewing the model answer.
MY_EVIDENCE = ""  # Include a value you observed and the report or cell that produced it.
reflection_path = record_reflection(ROOT, DAY, 'A candidate contains 82 rows and fails validation because seven rows are invalid. What must happen before the remaining 75 rows can be approved?', MY_ANSWER, MY_EVIDENCE)
print("Your reflection:", reflection_path.relative_to(ROOT))
if not MY_ANSWER.strip():
    print("Reflection pending: complete the answer in the file before submitting.")

<details><summary>Model answer: open after your attempt</summary>

Reject the complete candidate, quarantine the seven invalid rows with their reasons, and validate a new 75-row candidate. Approve the new candidate only when its quality checks pass.

Keep your first answer. Add a short correction in your reflection file if discussion changed your understanding.

</details>

## Save this day

1. Run the save cell below after all Spark sessions have stopped. It creates two packages.
2. In Colab use **File > Save a copy in GitHub**, select your repository and save this notebook as `day04/STUDENT.ipynb`. Keep your outputs.
3. Download the small `day04_github.zip`, unzip it on your computer, and upload its `student` and updated `day02/dbt` files to the same repository. Use a commit message such as `Complete Day 4`.
4. Keep the full handoff ZIP on Drive for your next session. It includes all cumulative Delta files and the separate Day 2 dbt workspace.
5. Open GitHub and confirm the notebook, reflection and results are visible.

Saving a notebook does not save Colab runtime files. The helper never publishes automatically and never asks you to put a token in a cell.

Before this cell, save your edits to the lab notes, shared project notes and reflection. Files marked **PENDING** still need your own explanation and observed evidence. Do not submit the unchanged prompts as completed work.

In [ ]:
saved = save_day(ROOT, DAY, drive_folder=DRIVE_FOLDER)
print(json.dumps(saved, indent=2))
print("Next: save this executed notebook to your GitHub repository using the File menu.")
if IS_COLAB and not NONINTERACTIVE:
    from google.colab import files
    files.download(saved["github_bundle"])